# 7장. RAG 시스템 평가 — 08. 온라인 평가 (Online Evaluation)

## 학습 목표

- **Offline 평가**와 **Online 평가(온라인 평가)**의 차이와 각각의 필요성 이해
- LangSmith Tracing으로 프로덕션 요청을 자동 로깅
- Latency(지연 시간), Token 사용량, Error Rate 등 실시간 모니터링 지표 이해

## 사전 지식

- 03~07번 노트북에서 Offline 평가 경험
- LangSmith 환경 설정

## Offline vs Online 평가 비교

```mermaid
flowchart LR
    subgraph Offline["🧪 Offline 평가 (개발 단계)"]
        direction TB
        O1[테스트 데이터셋<br/>준비]
        O2[evaluate 실행]
        O3[지표 점수 확인]
        O4[모델/파라미터<br/>개선]
        O1 --> O2 --> O3 --> O4 --> O1
    end

    subgraph Online["🌐 Online 평가 (프로덕션)"]
        direction TB
        N1[실제 사용자<br/>요청]
        N2[자동 Tracing<br/>LangSmith]
        N3[실시간<br/>모니터링]
        N4[이상 감지<br/>및 알림]
        N1 --> N2 --> N3 --> N4
    end

    Offline -->|배포| Online

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460

    class O1,N1 input
    class O2,O3,N2,N3 process
    class O4,N4 output
```

두 가지 평가 방식은 **상호 보완적**이에요. Offline으로 검증한 후 배포하고, Online으로 실제 운영 품질을 모니터링합니다.

---

## LangSmith Tracing 활성화

`LANGCHAIN_TRACING_V2=true`로 설정하면 이후 모든 LangChain 체인 실행이 자동으로 LangSmith에 기록돼요. 코드를 전혀 수정하지 않아도 됩니다.

In [ ]:
# ---------------------------------------------------
# LangSmith Tracing 활성화 — 이후 모든 체인 실행이 자동 로깅됨
# ---------------------------------------------------
from dotenv import load_dotenv
import os

load_dotenv()

# LANGCHAIN_TRACING_V2=true: LangChain 코드 변경 없이 자동 추적 시작
os.environ["LANGCHAIN_TRACING_V2"] = "true"
# LANGCHAIN_PROJECT: LangSmith 대시보드에서 구분할 프로젝트 이름
os.environ["LANGCHAIN_PROJECT"] = "production-rag-monitoring"

print("✅ LangSmith Tracing 활성화")
print(f"Project: {os.getenv('LANGCHAIN_PROJECT')}")

---

## RAG 시스템 (자동 로깅)

Tracing이 활성화된 상태에서 구축한 RAG 시스템은 모든 요청이 자동으로 LangSmith에 기록돼요. 별도의 평가 코드 없이도 트레이스(trace)가 남습니다.

> **실무 팁**: 프로덕션 배포 시 환경 변수 `LANGCHAIN_TRACING_V2=true`만 설정하면 모든 LangChain 기반 코드의 실행이 자동으로 추적됩니다. 비용이 발생하므로 샘플링 비율은 필요에 따라 조정하세요.

In [ ]:
# ---------------------------------------------------
# RAG 시스템 구축 (Tracing 활성화 상태에서 자동 로깅)
# ---------------------------------------------------
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계: 문서 로드 및 분할
loader = PyMuPDFLoader("data/attention_paper.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)

# 2단계: 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """Answer based on context.

Context: {context}
Question: {question}
Answer:"""
)

# 4단계: LLM 및 체인 생성
# Tracing이 활성화된 상태에서 체인을 실행하면 LangSmith가 자동으로 로깅함
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG 시스템 구축 완료 (LangSmith가 자동 로깅)")

---

## 시뮬레이션: 프로덕션 트래픽

실제 사용자 요청을 시뮬레이션해서 LangSmith에 트레이스를 남겨볼게요. 이후 대시보드에서 각 요청의 Latency, Token 사용량, 전체 실행 과정을 확인할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 프로덕션 트래픽 시뮬레이션 — 실행 결과가 LangSmith에 자동 기록됨
# ---------------------------------------------------
# ============================================================
# TODO: user_questions 리스트의 각 질문에 rag_chain.invoke()를 실행하세요
# 힌트: for 루프로 각 질문에 chain.invoke(question) 호출
# 예상 결과: 5개 질문에 대한 답변 출력 + LangSmith에 트레이스 기록
# ============================================================

# 시뮬레이션: 여러 사용자 요청
user_questions = [
    "What is the Transformer architecture?",
    "How does self-attention work?",
    "What are the advantages of Transformers?",
    "Explain multi-head attention",
    "What is the role of positional encoding?"
]



---

## 7장 전체 마무리

### RAG 시스템 평가 전체 흐름 요약

7장에서 배운 모든 평가 방법을 한눈에 정리해볼게요.

```mermaid
flowchart TD
    A[RAG 시스템 구축<br/>6장] --> B[01. 테스트 데이터셋 생성<br/>RAGAS 합성 데이터]
    B --> C[02. RAGAS 평가<br/>4가지 지표]
    C --> D[03. LangSmith<br/>데이터셋 + LLM-as-Judge]
    D --> E[04. 커스텀 평가자<br/>규칙 / LLM / 임베딩]
    E --> F[05. Heuristic 평가<br/>ROUGE / BLEU / METEOR / SemScore]
    F --> G[06. Groundedness 평가<br/>할루시네이션 탐지]
    G --> H[07. 모델 비교<br/>A/B 테스트]
    H --> I[08. Online 평가<br/>실시간 모니터링]
    I --> J{품질 목표<br/>달성?}
    J -->|아니오| K[개선: 임베딩 / 청크 /<br/>프롬프트 / 모델]
    K --> C
    J -->|예| L[프로덕션 배포]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A input
    class B,C,D,E,F,G,H,I process
    class J,K error
    class L output
```

### 평가 방법 선택 가이드

| 상황 | 추천 방법 |
|------|---------|
| RAG 파이프라인 첫 평가 | RAGAS 4가지 지표 |
| 빠른 반복 개발 중 | 규칙 기반 커스텀 평가자 |
| 참조 답변이 있을 때 | ROUGE + SemScore (Heuristic) |
| 할루시네이션 우려 | Groundedness 평가 |
| 모델 교체 검토 | LangSmith 모델 비교 |
| 프로덕션 운영 중 | Online Tracing + 모니터링 |
| 종합 품질 보증 | 위 방법 조합 |

### 7장에서 배운 핵심 개념 정리

- **RAGAS**: RAG 특화 자동 평가 프레임워크 (합성 데이터 생성 + 4가지 지표)
- **LangSmith**: 클라우드 기반 평가 플랫폼 (데이터셋 관리 + 실험 추적 + 모니터링)
- **LLM-as-Judge**: LLM을 평가자로 활용하는 자동화 패턴
- **Heuristic 평가**: LLM 없이 통계적으로 품질 측정 (ROUGE, BLEU, METEOR, SemScore)
- **Groundedness**: 할루시네이션 탐지를 위한 근거성 평가
- **Offline vs Online**: 개발 단계 평가 vs 프로덕션 모니터링의 상호 보완적 관계

좋은 RAG 시스템은 한 번의 평가로 완성되지 않아요. 지속적인 평가와 개선의 순환이 품질을 높여갑니다.